FASE 2 — Data Understanding: Censo 2020

2.1 Primera celda Markdown

# Fase 2 — Data Understanding
## Construcción y validación de Censo de Población y Vivienda 2020 — ITER

### Proyecto
Modelo econométrico de atractividad comercial municipal en México.

### Objetivo de esta etapa

Explorar y validar la información del Censo de Población y Vivienda 2020 de INEGI, utilizando la base ITER.

La unidad original de la fuente contiene información a nivel localidad. Para el proyecto econométrico se requiere obtener una observación por municipio.

La variable principal que se obtendrá de esta fuente será:

`GRAPROES`

correspondiente al grado promedio de escolaridad de la población de 15 años y más.

Esta variable será posteriormente utilizada como:

`X2 = Grado promedio de escolaridad municipal`

Antes de realizar transformaciones se verificará:

- estructura interna del archivo;
- variables disponibles;
- cobertura geográfica;
- claves de entidad, municipio y localidad;
- registros municipales;
- valores faltantes;
- duplicados;
- definición oficial de `GRAPROES`.

En esta etapa todavía no se integrará Censo con las demás fuentes ni se realizarán regresiones.

In [13]:
#############  2.2 Librerías

from pathlib import Path
import zipfile
from io import BytesIO

import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

print("Librerías cargadas correctamente.")

Librerías cargadas correctamente.


In [14]:
###########  
###########  2.3 Rutas
PROJECT_ROOT = Path.cwd().parent

CENSO_DIR = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "censo"
)

archivo_censo = (
    CENSO_DIR
    / "iter_00_cpv2020_csv.zip"
)

print("Ruta del proyecto:")
print(PROJECT_ROOT)

print("\nRuta Censo 2020:")
print(archivo_censo)

print("\n¿Existe el archivo?:", archivo_censo.exists())


Ruta del proyecto:
c:\Users\nashe\Desktop\Econometría\PROYECTO\atractividad_comercial_Mexico

Ruta Censo 2020:
c:\Users\nashe\Desktop\Econometría\PROYECTO\atractividad_comercial_Mexico\data\raw\censo\iter_00_cpv2020_csv.zip

¿Existe el archivo?: True


### 2.4 Inspección de la estructura del paquete ITER 2020, revisión del ZIP

Antes de cargar la base se revisa el contenido interno del archivo comprimido.

El objetivo es identificar:

- conjunto principal de datos;
- diccionario de variables;
- catálogo de tamaño de localidad;
- archivo de metadatos.

Esta revisión permite trabajar directamente con los archivos oficiales contenidos en el paquete.

In [15]:
with zipfile.ZipFile(archivo_censo, "r") as z:

    contenido_censo = z.namelist()

    print(
        f"Número de archivos encontrados: "
        f"{len(contenido_censo)}\n"
    )

    for nombre in contenido_censo:
        print(nombre)

Número de archivos encontrados: 8

iter_00_cpv2020/catalogos/
iter_00_cpv2020/catalogos/tam_loc.csv.csv
iter_00_cpv2020/conjunto_de_datos/
iter_00_cpv2020/conjunto_de_datos/conjunto_de_datos_iter_00CSV20.csv
iter_00_cpv2020/diccionario_datos/
iter_00_cpv2020/diccionario_datos/diccionario_datos_iter_00CSV20.csv
iter_00_cpv2020/metadatos/
iter_00_cpv2020/metadatos/metadatos_iter_00_cpv2020.txt


In [16]:
### 2.5 Identificar automáticamente el CSV principal

with zipfile.ZipFile(archivo_censo, "r") as z:

    candidatos_datos = [
        nombre for nombre in z.namelist()
        if "conjunto_de_datos" in nombre.lower()
        and nombre.lower().endswith(".csv")
    ]

    print("Archivos candidatos:")
    
    for nombre in candidatos_datos:
        print(nombre)

Archivos candidatos:
iter_00_cpv2020/conjunto_de_datos/conjunto_de_datos_iter_00CSV20.csv


In [17]:
####  2.6 Leer únicamente el encabezado

with zipfile.ZipFile(archivo_censo, "r") as z:

    with z.open(csv_censo) as archivo:

        encabezado_censo = pd.read_csv(
            archivo,
            encoding="utf-8-sig",
            nrows=0
        )

print(
    "Número de columnas:",
    len(encabezado_censo.columns)
)

print("\nPrimeras columnas:")

for i, columna in enumerate(
    encabezado_censo.columns[:15],
    start=1
):
    print(f"{i:03d}. {columna}")

Número de columnas: 286

Primeras columnas:
001. ENTIDAD
002. NOM_ENT
003. MUN
004. NOM_MUN
005. LOC
006. NOM_LOC
007. LONGITUD
008. LATITUD
009. ALTITUD
010. POBTOT
011. POBFEM
012. POBMAS
013. P_0A2
014. P_0A2_F
015. P_0A2_M


In [18]:
##########2.7 Validar variables necesarias

variables_censo_requeridas = [
    "ENTIDAD",
    "MUN",
    "LOC",
    "NOM_ENT",
    "NOM_MUN",
    "POBTOT",
    "GRAPROES"
]

validacion_variables_censo = pd.DataFrame({
    "variable": variables_censo_requeridas,
    "disponible": [
        variable in encabezado_censo.columns
        for variable in variables_censo_requeridas
    ]
})

display(validacion_variables_censo)

variables_faltantes_censo = [
    variable
    for variable in variables_censo_requeridas
    if variable not in encabezado_censo.columns
]

print(
    "Variables faltantes:",
    variables_faltantes_censo
)

,variable,disponible
0,ENTIDAD,True
1,MUN,True
2,LOC,True
3,NOM_ENT,True
4,NOM_MUN,True
5,POBTOT,True
6,GRAPROES,True


Variables faltantes: []


### 2.8 Lectura controlada de una muestra

Antes de cargar completamente la base ITER 2020, se realiza una lectura controlada de los primeros registros.

El objetivo es verificar:

- estructura de las observaciones;
- formato de las claves de entidad, municipio y localidad;
- nombres geográficos;
- población total;
- grado promedio de escolaridad;
- forma en que se identifican los registros correspondientes al total municipal.

Las claves geográficas se leen como texto para conservar los ceros a la izquierda.L

In [19]:
with zipfile.ZipFile(archivo_censo, "r") as z:

    with z.open(csv_censo) as archivo:

        muestra_censo = pd.read_csv(
            archivo,
            encoding="utf-8-sig",
            nrows=10,
            dtype={
                "ENTIDAD": "string",
                "MUN": "string",
                "LOC": "string"
            }
        )

# Normalizar claves para conservar el formato oficial
muestra_censo["ENTIDAD"] = (
    muestra_censo["ENTIDAD"]
    .str.strip()
    .str.zfill(2)
)

muestra_censo["MUN"] = (
    muestra_censo["MUN"]
    .str.strip()
    .str.zfill(3)
)

muestra_censo["LOC"] = (
    muestra_censo["LOC"]
    .str.strip()
    .str.zfill(4)
)

print("Dimensiones de la muestra:")
print(muestra_censo.shape)

display(
    muestra_censo[
        [
            "ENTIDAD",
            "MUN",
            "LOC",
            "NOM_ENT",
            "NOM_MUN",
            "NOM_LOC",
            "POBTOT",
            "GRAPROES"
        ]
    ]
)

Dimensiones de la muestra:
(10, 286)


,ENTIDAD,MUN,LOC,NOM_ENT,NOM_MUN,NOM_LOC,POBTOT,GRAPROES
0,00,000,0000,Total nacional,Total nacional,Total nacional,126014024,9.74
1,00,000,9998,Total nacional,Total nacional,Localidades de una vivienda,250354,6.5
2,00,000,9999,Total nacional,Total nacional,Localidades de dos viviendas,147125,6.45
3,01,000,0000,Aguascalientes,Total de la entidad Aguascalientes,Total de la Entidad,1425607,10.35
4,01,000,9998,Aguascalientes,Total de la entidad Aguascalientes,Localidades de una vivienda,3697,8.14
5,01,000,9999,Aguascalientes,Total de la entidad Aguascalientes,Localidades de dos viviendas,3021,8.37
6,01,001,0000,Aguascalientes,Aguascalientes,Total del Municipio,948990,10.84
7,01,001,0001,Aguascalientes,Aguascalientes,Aguascalientes,863893,11.01
8,01,001,0094,Aguascalientes,Aguascalientes,Granja Adelita,5,*
9,01,001,0096,Aguascalientes,Aguascalientes,Agua Azul,41,8.42


### 2.9 Identificación de los registros municipales mediante LOC

La base ITER contiene información tanto de agregados geográficos como de localidades individuales.

Por esta razón, se revisa el campo `LOC` para determinar cómo se identifica el registro correspondiente al total municipal.

El objetivo es confirmar empíricamente que `LOC = 0000` corresponde al agregado municipal antes de utilizar este criterio para construir la base de una observación por municipio.

In [20]:
display(
    muestra_censo[
        [
            "ENTIDAD",
            "MUN",
            "LOC",
            "NOM_ENT",
            "NOM_MUN",
            "NOM_LOC",
            "POBTOT",
            "GRAPROES"
        ]
    ]
)

,ENTIDAD,MUN,LOC,NOM_ENT,NOM_MUN,NOM_LOC,POBTOT,GRAPROES
0,00,000,0000,Total nacional,Total nacional,Total nacional,126014024,9.74
1,00,000,9998,Total nacional,Total nacional,Localidades de una vivienda,250354,6.5
2,00,000,9999,Total nacional,Total nacional,Localidades de dos viviendas,147125,6.45
3,01,000,0000,Aguascalientes,Total de la entidad Aguascalientes,Total de la Entidad,1425607,10.35
4,01,000,9998,Aguascalientes,Total de la entidad Aguascalientes,Localidades de una vivienda,3697,8.14
5,01,000,9999,Aguascalientes,Total de la entidad Aguascalientes,Localidades de dos viviendas,3021,8.37
6,01,001,0000,Aguascalientes,Aguascalientes,Total del Municipio,948990,10.84
7,01,001,0001,Aguascalientes,Aguascalientes,Aguascalientes,863893,11.01
8,01,001,0094,Aguascalientes,Aguascalientes,Granja Adelita,5,*
9,01,001,0096,Aguascalientes,Aguascalientes,Agua Azul,41,8.42


In [21]:
###### Después hacemos una comprobación específica del posible total municipal:

registros_loc_0000 = muestra_censo[
    muestra_censo["LOC"] == "0000"
]

print(
    f"Registros con LOC = 0000 en la muestra: "
    f"{len(registros_loc_0000)}"
)

display(
    registros_loc_0000[
        [
            "ENTIDAD",
            "MUN",
            "LOC",
            "NOM_ENT",
            "NOM_MUN",
            "NOM_LOC",
            "POBTOT",
            "GRAPROES"
        ]
    ]
)


Registros con LOC = 0000 en la muestra: 3


,ENTIDAD,MUN,LOC,NOM_ENT,NOM_MUN,NOM_LOC,POBTOT,GRAPROES
0,00,000,0000,Total nacional,Total nacional,Total nacional,126014024,9.74
3,01,000,0000,Aguascalientes,Total de la entidad Aguascalientes,Total de la Entidad,1425607,10.35
6,01,001,0000,Aguascalientes,Aguascalientes,Total del Municipio,948990,10.84


### 2.10 Revisión del diccionario oficial ITER 2020

Se revisa el diccionario de datos incluido en el paquete oficial del Censo de Población y Vivienda 2020.

El objetivo es documentar las variables utilizadas para identificar los registros municipales y construir la variable de escolaridad:

- `ENTIDAD`
- `MUN`
- `LOC`
- `NOM_ENT`
- `NOM_MUN`
- `POBTOT`
- `GRAPROES`

Esta revisión permite fundamentar la selección de variables directamente en la documentación de la fuente.

In [22]:
with zipfile.ZipFile(archivo_censo, "r") as z:

    candidatos_diccionario = [
        nombre for nombre in z.namelist()
        if "diccionario" in nombre.lower()
        and nombre.lower().endswith(".csv")
    ]

    print("Archivos de diccionario encontrados:")

    for nombre in candidatos_diccionario:
        print(nombre)

Archivos de diccionario encontrados:
iter_00_cpv2020/diccionario_datos/diccionario_datos_iter_00CSV20.csv


In [24]:
# Identificar el archivo de diccionario
with zipfile.ZipFile(archivo_censo, "r") as z:

    candidatos_diccionario = [
        nombre for nombre in z.namelist()
        if "diccionario" in nombre.lower()
        and nombre.lower().endswith(".csv")
    ]

    diccionario_censo_csv = candidatos_diccionario[0]

    # El encabezado real se encuentra en la fila 5 del archivo
    # (índice 4 para Pandas)
    with z.open(diccionario_censo_csv) as archivo:

        diccionario_censo = pd.read_csv(
            archivo,
            encoding="utf-8-sig",
            header=4
        )

# Eliminar columnas completamente vacías
diccionario_censo = (
    diccionario_censo
    .dropna(axis=1, how="all")
    .copy()
)

# Limpiar espacios en nombres de columnas
diccionario_censo.columns = (
    diccionario_censo.columns
    .astype(str)
    .str.strip()
)

# Limpiar espacios en el campo Mnemónico
diccionario_censo["Mnemónico"] = (
    diccionario_censo["Mnemónico"]
    .astype("string")
    .str.strip()
)

print("Dimensiones del diccionario limpio:")
print(diccionario_censo.shape)

print("\nColumnas:")
print(diccionario_censo.columns.tolist())

display(diccionario_censo.head(15))

Dimensiones del diccionario limpio:
(286, 7)

Columnas:
['Núm.', 'Indicador', 'Descripción', 'Mnemónico', 'Rangos', 'Longitud', 'Unnamed: 9']


,Núm.,Indicador,Descripción,Mnemónico,Rangos,Longitud,Unnamed: 9
0,1,Clave de entidad federativa,Código que identifica a la entidad federativa....,ENTIDAD,00…32,2,NaN
1,2,Entidad federativa,Nombre oficial de la entidad federativa.,NOM_ENT,Alfanumérico,50,NaN
2,3,Clave de municipio o demarcación territorial,Código que identifica al municipio o demarcaci...,MUN,000…570,3,
3,4,Municipio o demarcación territorial,Nombre oficial del municipio o demarcación ter...,NOM_MUN,Alfanumérico,50,NaN
4,5,Clave de localidad,Código que identifica a la localidad al interi...,LOC,0000…9999,4,NaN
5,6,Localidad,Nombre con el que se reconoce a la localidad d...,NOM_LOC,Alfanumérico,70,NaN
6,7,Longitud,"Ubicación de la localidad expresada en grados,...",LONGITUD,Caracter,16,NaN
7,8,Latitud,"Ubicación de la localidad expresada en grados,...",LATITUD,Caracter,15,NaN
8,9,Altitud,"Altura a la que se encuentra una localidad, re...",ALTITUD,Caracter,4,NaN
9,1,Población total,Total de personas que residen habitualmente en...,POBTOT,0...999999999,9,NaN


In [25]:
############2.10 — Limpieza final del diccionario

# Eliminar columnas residuales tipo "Unnamed"
diccionario_censo = diccionario_censo.loc[
    :,
    ~diccionario_censo.columns.str.startswith("Unnamed")
].copy()

print("Dimensiones finales del diccionario:")
print(diccionario_censo.shape)

print("\nColumnas finales:")
print(diccionario_censo.columns.tolist())

display(diccionario_censo.head(15))

Dimensiones finales del diccionario:
(286, 6)

Columnas finales:
['Núm.', 'Indicador', 'Descripción', 'Mnemónico', 'Rangos', 'Longitud']


,Núm.,Indicador,Descripción,Mnemónico,Rangos,Longitud
0,1,Clave de entidad federativa,Código que identifica a la entidad federativa....,ENTIDAD,00…32,2
1,2,Entidad federativa,Nombre oficial de la entidad federativa.,NOM_ENT,Alfanumérico,50
2,3,Clave de municipio o demarcación territorial,Código que identifica al municipio o demarcaci...,MUN,000…570,3
3,4,Municipio o demarcación territorial,Nombre oficial del municipio o demarcación ter...,NOM_MUN,Alfanumérico,50
4,5,Clave de localidad,Código que identifica a la localidad al interi...,LOC,0000…9999,4
5,6,Localidad,Nombre con el que se reconoce a la localidad d...,NOM_LOC,Alfanumérico,70
6,7,Longitud,"Ubicación de la localidad expresada en grados,...",LONGITUD,Caracter,16
7,8,Latitud,"Ubicación de la localidad expresada en grados,...",LATITUD,Caracter,15
8,9,Altitud,"Altura a la que se encuentra una localidad, re...",ALTITUD,Caracter,4
9,1,Población total,Total de personas que residen habitualmente en...,POBTOT,0...999999999,9


2.11 Construcción del diccionario técnico del proyecto


### 2.11 Diccionario técnico de variables Censo 2020

A partir del diccionario oficial del Censo de Población y Vivienda 2020 se seleccionan las variables necesarias para construir la base municipal del proyecto.

Las variables conservadas permiten:

- identificar entidad y municipio;
- distinguir los registros correspondientes al total municipal;
- validar la población total;
- obtener el grado promedio de escolaridad municipal.

La variable `GRAPROES` será utilizada posteriormente como la variable explicativa X2 del modelo econométrico.


In [26]:
variables_diccionario_censo = [
    "ENTIDAD",
    "NOM_ENT",
    "MUN",
    "NOM_MUN",
    "LOC",
    "POBTOT",
    "GRAPROES"
]

diccionario_tecnico_censo = (
    diccionario_censo[
        diccionario_censo["Mnemónico"]
        .isin(variables_diccionario_censo)
    ]
    .copy()
)

# Ordenar según la lista definida
orden_variables = {
    variable: i
    for i, variable in enumerate(variables_diccionario_censo)
}

diccionario_tecnico_censo["orden"] = (
    diccionario_tecnico_censo["Mnemónico"]
    .map(orden_variables)
)

diccionario_tecnico_censo = (
    diccionario_tecnico_censo
    .sort_values("orden")
    .drop(columns="orden")
    .reset_index(drop=True)
)

display(diccionario_tecnico_censo)

,Núm.,Indicador,Descripción,Mnemónico,Rangos,Longitud
0,1,Clave de entidad federativa,Código que identifica a la entidad federativa....,ENTIDAD,00…32,2
1,2,Entidad federativa,Nombre oficial de la entidad federativa.,NOM_ENT,Alfanumérico,50
2,3,Clave de municipio o demarcación territorial,Código que identifica al municipio o demarcaci...,MUN,000…570,3
3,4,Municipio o demarcación territorial,Nombre oficial del municipio o demarcación ter...,NOM_MUN,Alfanumérico,50
4,5,Clave de localidad,Código que identifica a la localidad al interi...,LOC,0000…9999,4
5,1,Población total,Total de personas que residen habitualmente en...,POBTOT,0...999999999,9
6,186,Grado promedio de escolaridad,Resultado de dividir el monto de grados escola...,GRAPROES,0...999999999,9


In [27]:
######## 2.12 Validar que estén las siete

variables_encontradas_censo = set(
    diccionario_tecnico_censo["Mnemónico"]
)

variables_faltantes_diccionario = [
    variable
    for variable in variables_diccionario_censo
    if variable not in variables_encontradas_censo
]

print(
    "Variables solicitadas:",
    len(variables_diccionario_censo)
)

print(
    "Variables encontradas:",
    len(variables_encontradas_censo)
)

print(
    "Variables faltantes:",
    variables_faltantes_diccionario
)

Variables solicitadas: 7
Variables encontradas: 7
Variables faltantes: []


In [28]:
#########   2.13 Ver específicamente la definición oficial de GRAPROES

definicion_graproes = (
    diccionario_tecnico_censo[
        diccionario_tecnico_censo["Mnemónico"] == "GRAPROES"
    ]
)

display(definicion_graproes)

,Núm.,Indicador,Descripción,Mnemónico,Rangos,Longitud
6,186,Grado promedio de escolaridad,Resultado de dividir el monto de grados escola...,GRAPROES,0...999999999,9


2.14 Carga controlada de la base ITER 2020


### 2.14 Carga controlada de la base ITER 2020

Una vez validadas las variables y su definición en el diccionario oficial, se carga la base completa utilizando únicamente las columnas necesarias para el proyecto.

La selección reducida evita cargar las 286 variables disponibles cuando el objetivo actual es construir la información municipal de escolaridad.

Se conservarán:

- claves de entidad, municipio y localidad;
- nombres geográficos;
- población total;
- grado promedio de escolaridad.

In [29]:
columnas_censo_interes = [
    "ENTIDAD",
    "MUN",
    "LOC",
    "NOM_ENT",
    "NOM_MUN",
    "NOM_LOC",
    "POBTOT",
    "GRAPROES"
]

with zipfile.ZipFile(archivo_censo, "r") as z:

    with z.open(csv_censo) as archivo:

        censo_raw = pd.read_csv(
            archivo,
            encoding="utf-8-sig",
            usecols=columnas_censo_interes,
            dtype={
                "ENTIDAD": "string",
                "MUN": "string",
                "LOC": "string",
                "GRAPROES": "string"
            }
        )

print("Dimensiones de la base cargada:")
print(censo_raw.shape)

display(censo_raw.head(10))

Dimensiones de la base cargada:
(195662, 8)


,ENTIDAD,NOM_ENT,MUN,NOM_MUN,LOC,NOM_LOC,POBTOT,GRAPROES
0,00,Total nacional,000,Total nacional,0000,Total nacional,126014024,9.74
1,00,Total nacional,000,Total nacional,9998,Localidades de una vivienda,250354,6.5
2,00,Total nacional,000,Total nacional,9999,Localidades de dos viviendas,147125,6.45
3,01,Aguascalientes,000,Total de la entidad Aguascalientes,0000,Total de la Entidad,1425607,10.35
4,01,Aguascalientes,000,Total de la entidad Aguascalientes,9998,Localidades de una vivienda,3697,8.14
5,01,Aguascalientes,000,Total de la entidad Aguascalientes,9999,Localidades de dos viviendas,3021,8.37
6,01,Aguascalientes,001,Aguascalientes,0000,Total del Municipio,948990,10.84
7,01,Aguascalientes,001,Aguascalientes,0001,Aguascalientes,863893,11.01
8,01,Aguascalientes,001,Aguascalientes,0094,Granja Adelita,5,*
9,01,Aguascalientes,001,Aguascalientes,0096,Agua Azul,41,8.42


### 2.15 Normalización de claves geográficas

Las claves de entidad, municipio y localidad se normalizan para conservar el número oficial de posiciones y los ceros a la izquierda.

Esta estandarización permitirá construir posteriormente la llave municipal `CVEGEO`.

In [30]:
censo_raw["ENTIDAD"] = (
    censo_raw["ENTIDAD"]
    .str.strip()
    .str.zfill(2)
)

censo_raw["MUN"] = (
    censo_raw["MUN"]
    .str.strip()
    .str.zfill(3)
)

censo_raw["LOC"] = (
    censo_raw["LOC"]
    .str.strip()
    .str.zfill(4)
)

display(
    censo_raw[
        [
            "ENTIDAD",
            "MUN",
            "LOC",
            "NOM_ENT",
            "NOM_MUN",
            "NOM_LOC"
        ]
    ].head(10)
)

,ENTIDAD,MUN,LOC,NOM_ENT,NOM_MUN,NOM_LOC
0,00,000,0000,Total nacional,Total nacional,Total nacional
1,00,000,9998,Total nacional,Total nacional,Localidades de una vivienda
2,00,000,9999,Total nacional,Total nacional,Localidades de dos viviendas
3,01,000,0000,Aguascalientes,Total de la entidad Aguascalientes,Total de la Entidad
4,01,000,9998,Aguascalientes,Total de la entidad Aguascalientes,Localidades de una vivienda
5,01,000,9999,Aguascalientes,Total de la entidad Aguascalientes,Localidades de dos viviendas
6,01,001,0000,Aguascalientes,Aguascalientes,Total del Municipio
7,01,001,0001,Aguascalientes,Aguascalientes,Aguascalientes
8,01,001,0094,Aguascalientes,Aguascalientes,Granja Adelita
9,01,001,0096,Aguascalientes,Aguascalientes,Agua Azul


### 2.16 Selección de registros municipales

La base ITER contiene totales nacionales, estatales, municipales y registros individuales de localidades.

Para conservar exclusivamente una observación por municipio se aplican simultáneamente los siguientes criterios:

`ENTIDAD != 00`

`MUN != 000`

`LOC == 0000`

De esta manera se excluyen el total nacional, los totales estatales y las localidades individuales.

In [31]:
censo_municipal_raw = censo_raw[
    (censo_raw["ENTIDAD"] != "00")
    & (censo_raw["MUN"] != "000")
    & (censo_raw["LOC"] == "0000")
].copy()

print(
    "Registros municipales encontrados:",
    f"{len(censo_municipal_raw):,}"
)

print(
    "Entidades representadas:",
    censo_municipal_raw["ENTIDAD"].nunique()
)

display(
    censo_municipal_raw[
        [
            "ENTIDAD",
            "MUN",
            "LOC",
            "NOM_ENT",
            "NOM_MUN",
            "NOM_LOC",
            "POBTOT",
            "GRAPROES"
        ]
    ].head(15)
)

Registros municipales encontrados: 2,469
Entidades representadas: 32


,ENTIDAD,MUN,LOC,NOM_ENT,NOM_MUN,NOM_LOC,POBTOT,GRAPROES
6,01,001,0000,Aguascalientes,Aguascalientes,Total del Municipio,948990,10.84
574,01,002,0000,Aguascalientes,Asientos,Total del Municipio,51536,8.54
749,01,003,0000,Aguascalientes,Calvillo,Total del Municipio,58250,8.05
918,01,004,0000,Aguascalientes,Cosío,Total del Municipio,17000,9.08
985,01,005,0000,Aguascalientes,Jesús María,Total del Municipio,129929,10.22
1203,01,006,0000,Aguascalientes,Pabellón de Arteaga,Total del Municipio,47646,9.77
1397,01,007,0000,Aguascalientes,Rincón de Romos,Total del Municipio,57369,9.6
1654,01,008,0000,Aguascalientes,San José de Gracia,Total del Municipio,9552,9.24
1689,01,009,0000,Aguascalientes,Tepezalá,Total del Municipio,22485,8.56
1793,01,010,0000,Aguascalientes,El Llano,Total del Municipio,20853,8.49


### 2.17 Validación de GRAPROES a nivel municipal

Antes de convertir `GRAPROES` a formato numérico se verifica si los registros correspondientes a totales municipales contienen símbolos de confidencialidad, valores no disponibles u otras cadenas no numéricas.

Los valores no numéricos no serán sustituidos por cero.

In [32]:
graproes_original = (
    censo_municipal_raw["GRAPROES"]
    .astype("string")
    .str.strip()
)

print(
    "Asteriscos en GRAPROES municipal:",
    (graproes_original == "*").sum()
)

print(
    "Valores nulos originales:",
    graproes_original.isna().sum()
)

Asteriscos en GRAPROES municipal: 0
Valores nulos originales: 0


In [33]:
#################Ahora localizamos cualquier valor no numérico

es_numerico = graproes_original.str.match(
    r"^\d+(\.\d+)?$",
    na=False
)

valores_no_numericos = (
    graproes_original[
        ~es_numerico
        & graproes_original.notna()
    ]
    .value_counts()
)

print("Valores no numéricos encontrados:")
display(valores_no_numericos)

Valores no numéricos encontrados:


Series([], Name: count, dtype: Int64)

In [34]:
###  2.18 — Construcción de CVEGEO

censo_municipal_raw["CVEGEO"] = (
    censo_municipal_raw["ENTIDAD"]
    + censo_municipal_raw["MUN"]
)

print(
    "CVEGEO únicos:",
    censo_municipal_raw["CVEGEO"].nunique()
)

print(
    "CVEGEO duplicados:",
    censo_municipal_raw["CVEGEO"]
    .duplicated()
    .sum()
)

display(
    censo_municipal_raw[
        [
            "CVEGEO",
            "NOM_ENT",
            "NOM_MUN",
            "GRAPROES"
        ]
    ].head(10)
)

CVEGEO únicos: 2469
CVEGEO duplicados: 0


,CVEGEO,NOM_ENT,NOM_MUN,GRAPROES
6,01001,Aguascalientes,Aguascalientes,10.84
574,01002,Aguascalientes,Asientos,8.54
749,01003,Aguascalientes,Calvillo,8.05
918,01004,Aguascalientes,Cosío,9.08
985,01005,Aguascalientes,Jesús María,10.22
1203,01006,Aguascalientes,Pabellón de Arteaga,9.77
1397,01007,Aguascalientes,Rincón de Romos,9.6
1654,01008,Aguascalientes,San José de Gracia,9.24
1689,01009,Aguascalientes,Tepezalá,8.56
1793,01010,Aguascalientes,El Llano,8.49


In [35]:
###################   2.19 — Convertir GRAPROES a numérico

censo_municipal_raw["GRAPROES"] = pd.to_numeric(
    censo_municipal_raw["GRAPROES"],
    errors="coerce"
)

print(
    "GRAPROES nulos después de convertir:",
    censo_municipal_raw["GRAPROES"].isna().sum()
)

print(
    "GRAPROES mínimo:",
    censo_municipal_raw["GRAPROES"].min()
)

print(
    "GRAPROES máximo:",
    censo_municipal_raw["GRAPROES"].max()
)

GRAPROES nulos después de convertir: 0
GRAPROES mínimo: 3.4
GRAPROES máximo: 14.55


### 2.20 Conclusión de Data Understanding — Censo 2020

La revisión de la base ITER del Censo de Población y Vivienda 2020 permitió confirmar la disponibilidad y calidad de la información necesaria para construir la variable educativa del proyecto.

Principales resultados:

- La base ITER contiene 286 variables.
- Se identificaron y documentaron las siete variables requeridas para el análisis.
- Se confirmó mediante el diccionario oficial la definición de `GRAPROES` como grado promedio de escolaridad.
- Se identificó correctamente la estructura geográfica de la base.
- Los registros municipales se seleccionan mediante los criterios:
  - `ENTIDAD != 00`
  - `MUN != 000`
  - `LOC == 0000`
- Los valores `*` observados en algunas localidades corresponden a información no publicada en unidades geográficas pequeñas y no afectan los registros municipales utilizados en el proyecto.
- A nivel municipal no se encontraron valores no numéricos en `GRAPROES`.
- La conversión de `GRAPROES` a formato numérico no generó valores faltantes.
- El grado promedio de escolaridad municipal presenta valores entre 3.40 y 14.55.

Con base en estos controles, la información se considera adecuada para construir la base municipal del Censo 2020 y la variable explicativa `GRAPROES_2020`.

FASE 3 — Data Preparation

# Fase 3 — Data Preparation
## Construcción de la base municipal Censo 2020

A partir de los registros correspondientes a totales municipales se construye una base con una sola observación por municipio.

Se conservarán:

- `CVEGEO`: clave geográfica municipal;
- entidad federativa;
- municipio;
- `POBTOT_CENSO_2020`: población total censal utilizada como variable de control;
- `GRAPROES_2020`: grado promedio de escolaridad municipal.

`GRAPROES_2020` será la variable explicativa X2 del modelo econométrico.

La población censal se conservará para validaciones posteriores, pero no sustituirá las poblaciones CONAPO utilizadas para construir la densidad comercial 2020–2025.

In [36]:
###############3.1 Construir la base municipal

censo_2020_municipal = (
    censo_municipal_raw[
        [
            "CVEGEO",
            "NOM_ENT",
            "NOM_MUN",
            "POBTOT",
            "GRAPROES"
        ]
    ]
    .copy()
    .rename(
        columns={
            "NOM_ENT": "entidad",
            "NOM_MUN": "municipio",
            "POBTOT": "POBTOT_CENSO_2020",
            "GRAPROES": "GRAPROES_2020"
        }
    )
    .sort_values("CVEGEO")
    .reset_index(drop=True)
)

print("Dimensiones de la base municipal:")
print(censo_2020_municipal.shape)

display(censo_2020_municipal.head(10))

Dimensiones de la base municipal:
(2469, 5)


,CVEGEO,entidad,municipio,POBTOT_CENSO_2020,GRAPROES_2020
0,01001,Aguascalientes,Aguascalientes,948990,10.84
1,01002,Aguascalientes,Asientos,51536,8.54
2,01003,Aguascalientes,Calvillo,58250,8.05
3,01004,Aguascalientes,Cosío,17000,9.08
4,01005,Aguascalientes,Jesús María,129929,10.22
5,01006,Aguascalientes,Pabellón de Arteaga,47646,9.77
6,01007,Aguascalientes,Rincón de Romos,57369,9.6
7,01008,Aguascalientes,San José de Gracia,9552,9.24
8,01009,Aguascalientes,Tepezalá,22485,8.56
9,01010,Aguascalientes,El Llano,20853,8.49


In [37]:
####3.2 Control de calidad

print("CONTROL DE CALIDAD — CENSO MUNICIPAL 2020")
print("=" * 55)

print(
    "Número de municipios:",
    f"{len(censo_2020_municipal):,}"
)

print(
    "CVEGEO únicos:",
    f"{censo_2020_municipal['CVEGEO'].nunique():,}"
)

print(
    "CVEGEO duplicados:",
    censo_2020_municipal["CVEGEO"]
    .duplicated()
    .sum()
)

print(
    "Valores nulos totales:",
    censo_2020_municipal
    .isna()
    .sum()
    .sum()
)

print(
    "POBTOT_CENSO_2020 <= 0:",
    (
        censo_2020_municipal["POBTOT_CENSO_2020"]
        <= 0
    ).sum()
)

print(
    "GRAPROES_2020 nulos:",
    censo_2020_municipal["GRAPROES_2020"]
    .isna()
    .sum()
)

print(
    "GRAPROES_2020 mínimo:",
    censo_2020_municipal["GRAPROES_2020"].min()
)

print(
    "GRAPROES_2020 máximo:",
    censo_2020_municipal["GRAPROES_2020"].max()
)

CONTROL DE CALIDAD — CENSO MUNICIPAL 2020
Número de municipios: 2,469
CVEGEO únicos: 2,469
CVEGEO duplicados: 0
Valores nulos totales: 0
POBTOT_CENSO_2020 <= 0: 0
GRAPROES_2020 nulos: 0
GRAPROES_2020 mínimo: 3.4
GRAPROES_2020 máximo: 14.55


In [38]:
###  3.3 Validaciones automáticas
assert (
    censo_2020_municipal["CVEGEO"].nunique()
    == len(censo_2020_municipal)
), "Existen problemas de unicidad en CVEGEO."

assert (
    censo_2020_municipal["CVEGEO"]
    .duplicated()
    .sum()
    == 0
), "Existen CVEGEO duplicados."

assert (
    censo_2020_municipal
    .isna()
    .sum()
    .sum()
    == 0
), "Existen valores faltantes."

assert (
    censo_2020_municipal["POBTOT_CENSO_2020"]
    .gt(0)
    .all()
), "Existen poblaciones censales no positivas."

assert (
    censo_2020_municipal["GRAPROES_2020"]
    .notna()
    .all()
), "Existen valores faltantes en GRAPROES_2020."

print(
    "Todas las validaciones fueron superadas correctamente."
)

Todas las validaciones fueron superadas correctamente.


### 3.4 Exportación de la base municipal Censo 2020

Después de superar los controles de calidad, la base municipal del Censo 2020 se guarda como archivo procesado.

La base contiene una observación por municipio y conserva:

- `CVEGEO`
- entidad federativa
- municipio
- población total censal 2020
- grado promedio de escolaridad 2020

La variable `GRAPROES_2020` será utilizada posteriormente como la variable explicativa X2 del modelo econométrico.

In [39]:
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

PROCESSED_DIR.mkdir(
    parents=True,
    exist_ok=True
)

archivo_salida_censo = (
    PROCESSED_DIR / "censo_2020_municipal.csv"
)

censo_2020_municipal.to_csv(
    archivo_salida_censo,
    index=False,
    encoding="utf-8-sig"
)

print("Base Censo 2020 guardada correctamente.")
print(f"Ruta: {archivo_salida_censo}")

Base Censo 2020 guardada correctamente.
Ruta: c:\Users\nashe\Desktop\Econometría\PROYECTO\atractividad_comercial_Mexico\data\processed\censo_2020_municipal.csv


In [40]:
###########  3.5 Verificar el archivo guardado

censo_verificacion = pd.read_csv(
    archivo_salida_censo,
    dtype={"CVEGEO": "string"}
)

print("Dimensiones:")
print(censo_verificacion.shape)

print(
    "CVEGEO únicos:",
    censo_verificacion["CVEGEO"].nunique()
)

print(
    "Valores nulos:",
    censo_verificacion.isna().sum().sum()
)

display(censo_verificacion.head(10))

Dimensiones:
(2469, 5)
CVEGEO únicos: 2469
Valores nulos: 0


,CVEGEO,entidad,municipio,POBTOT_CENSO_2020,GRAPROES_2020
0,01001,Aguascalientes,Aguascalientes,948990,10.84
1,01002,Aguascalientes,Asientos,51536,8.54
2,01003,Aguascalientes,Calvillo,58250,8.05
3,01004,Aguascalientes,Cosío,17000,9.08
4,01005,Aguascalientes,Jesús María,129929,10.22
5,01006,Aguascalientes,Pabellón de Arteaga,47646,9.77
6,01007,Aguascalientes,Rincón de Romos,57369,9.60
7,01008,Aguascalientes,San José de Gracia,9552,9.24
8,01009,Aguascalientes,Tepezalá,22485,8.56
9,01010,Aguascalientes,El Llano,20853,8.49


### 3.6 Auditoría de cobertura Censo vs. DENUE–CONAPO

Se compara la cobertura municipal del Censo 2020 con la muestra candidata obtenida previamente de DENUE 2020, DENUE 2025 y CONAPO.

El objetivo es identificar:

- municipios presentes en las cuatro fuentes;
- municipios candidatos sin correspondencia en Censo;
- municipios del Censo que todavía no forman parte de la muestra candidata.

La muestra resultante seguirá considerándose provisional, ya que aún faltan ILMM y CONEVAL.

In [41]:
#####Muestra candidata

archivo_muestra_candidata = (
    PROCESSED_DIR
    / "muestra_candidata_denue_conapo.csv"
)

muestra_candidata = pd.read_csv(
    archivo_muestra_candidata,
    dtype={"CVEGEO": "string"}
)

print(
    "Municipios candidatos DENUE–CONAPO:",
    f"{len(muestra_candidata):,}"
)

print(
    "Municipios Censo 2020:",
    f"{len(censo_2020_municipal):,}"
)

Municipios candidatos DENUE–CONAPO: 2,465
Municipios Censo 2020: 2,469


In [42]:
###############  3.7 Comparar las claves

claves_candidatas = set(
    muestra_candidata["CVEGEO"]
)

claves_censo = set(
    censo_2020_municipal["CVEGEO"]
)

claves_comunes_censo = (
    claves_candidatas
    & claves_censo
)

solo_candidata = (
    claves_candidatas
    - claves_censo
)

solo_censo = (
    claves_censo
    - claves_candidatas
)

print(
    "Municipios comunes:",
    f"{len(claves_comunes_censo):,}"
)

print(
    "Candidatos sin Censo:",
    f"{len(solo_candidata):,}"
)

print(
    "Censo fuera de muestra candidata:",
    f"{len(solo_censo):,}"
)

Municipios comunes: 2,465
Candidatos sin Censo: 0
Censo fuera de muestra candidata: 4


In [43]:
###########  3.8 Identificar cualquier municipio faltante.  Candidatos DENUE–CONAPO que no aparezcan en Censo

detalle_candidatos_sin_censo = (
    muestra_candidata[
        muestra_candidata["CVEGEO"]
        .isin(solo_candidata)
    ]
    .sort_values("CVEGEO")
    .reset_index(drop=True)
)

print(
    "Candidatos sin correspondencia en Censo:",
    len(detalle_candidatos_sin_censo)
)

display(detalle_candidatos_sin_censo)

Candidatos sin correspondencia en Censo: 0


,CVEGEO,DENUE_2020,DENUE_2025,CONAPO


In [44]:
#############   3.9 Construir la nueva muestra candidata

muestra_candidata_censo = (
    muestra_candidata[
        muestra_candidata["CVEGEO"]
        .isin(claves_comunes_censo)
    ]
    .copy()
    .sort_values("CVEGEO")
    .reset_index(drop=True)
)

print(
    "Nueva muestra candidata:",
    f"{len(muestra_candidata_censo):,}"
)

Nueva muestra candidata: 2,465


In [45]:
####  guardamos

archivo_muestra_candidata_censo = (
    PROCESSED_DIR
    / "muestra_candidata_denue_conapo_censo.csv"
)

muestra_candidata_censo.to_csv(
    archivo_muestra_candidata_censo,
    index=False,
    encoding="utf-8-sig"
)

print("Nueva muestra candidata guardada.")

Nueva muestra candidata guardada.


In [46]:
####  1.Guardar la base municipal del Censo 2020.

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

PROCESSED_DIR.mkdir(
    parents=True,
    exist_ok=True
)

archivo_salida_censo = (
    PROCESSED_DIR / "censo_2020_municipal.csv"
)

censo_2020_municipal.to_csv(
    archivo_salida_censo,
    index=False,
    encoding="utf-8-sig"
)

print("Base Censo 2020 guardada correctamente.")
print(f"Ruta: {archivo_salida_censo}")

Base Censo 2020 guardada correctamente.
Ruta: c:\Users\nashe\Desktop\Econometría\PROYECTO\atractividad_comercial_Mexico\data\processed\censo_2020_municipal.csv


In [47]:
censo_verificacion = pd.read_csv(
    archivo_salida_censo,
    dtype={"CVEGEO": "string"}
)

print("Dimensiones:", censo_verificacion.shape)

print(
    "CVEGEO únicos:",
    censo_verificacion["CVEGEO"].nunique()
)

print(
    "Valores nulos:",
    censo_verificacion.isna().sum().sum()
)

display(censo_verificacion.head(10))

Dimensiones: (2469, 5)
CVEGEO únicos: 2469
Valores nulos: 0


,CVEGEO,entidad,municipio,POBTOT_CENSO_2020,GRAPROES_2020
0,01001,Aguascalientes,Aguascalientes,948990,10.84
1,01002,Aguascalientes,Asientos,51536,8.54
2,01003,Aguascalientes,Calvillo,58250,8.05
3,01004,Aguascalientes,Cosío,17000,9.08
4,01005,Aguascalientes,Jesús María,129929,10.22
5,01006,Aguascalientes,Pabellón de Arteaga,47646,9.77
6,01007,Aguascalientes,Rincón de Romos,57369,9.60
7,01008,Aguascalientes,San José de Gracia,9552,9.24
8,01009,Aguascalientes,Tepezalá,22485,8.56
9,01010,Aguascalientes,El Llano,20853,8.49


2. Auditar Censo contra nuestra muestra candidata DENUE–CONAPO.

### 3.5 Auditoría de cobertura Censo vs. DENUE–CONAPO

Se compara la cobertura municipal del Censo 2020 con la muestra candidata construida previamente a partir de DENUE 2020, DENUE 2025 y CONAPO.

El objetivo es identificar:

- municipios presentes en las cuatro fuentes;
- municipios de la muestra candidata sin información censal;
- municipios del Censo que no pertenecen todavía a la muestra candidata.

La muestra resultante continúa siendo provisional, ya que posteriormente deberán incorporarse ILMM y CONEVAL.

In [48]:
#################### Muestra candidata
archivo_muestra_candidata = (
    PROCESSED_DIR
    / "muestra_candidata_denue_conapo.csv"
)

muestra_candidata = pd.read_csv(
    archivo_muestra_candidata,
    dtype={"CVEGEO": "string"}
)

print(
    "Municipios candidatos DENUE–CONAPO:",
    f"{len(muestra_candidata):,}"
)

print(
    "Municipios Censo 2020:",
    f"{len(censo_2020_municipal):,}"
)

Municipios candidatos DENUE–CONAPO: 2,465
Municipios Censo 2020: 2,469


In [49]:
#### compara las claves

claves_candidatas = set(
    muestra_candidata["CVEGEO"]
)

claves_censo = set(
    censo_2020_municipal["CVEGEO"]
)

claves_comunes_censo = (
    claves_candidatas
    & claves_censo
)

solo_candidata = (
    claves_candidatas
    - claves_censo
)

solo_censo = (
    claves_censo
    - claves_candidatas
)

print(
    "Municipios comunes:",
    f"{len(claves_comunes_censo):,}"
)

print(
    "Candidatos sin Censo:",
    f"{len(solo_candidata):,}"
)

print(
    "Censo fuera de muestra candidata:",
    f"{len(solo_censo):,}"
)

Municipios comunes: 2,465
Candidatos sin Censo: 0
Censo fuera de muestra candidata: 4


In [50]:
#########  identifica exactamente las diferencias
detalle_candidatos_sin_censo = (
    muestra_candidata[
        muestra_candidata["CVEGEO"]
        .isin(solo_candidata)
    ]
    .sort_values("CVEGEO")
    .reset_index(drop=True)
)

print(
    "Candidatos sin correspondencia en Censo:",
    len(detalle_candidatos_sin_censo)
)

display(detalle_candidatos_sin_censo)

Candidatos sin correspondencia en Censo: 0


,CVEGEO,DENUE_2020,DENUE_2025,CONAPO


In [51]:
detalle_censo_fuera_candidata = (
    censo_2020_municipal[
        censo_2020_municipal["CVEGEO"]
        .isin(solo_censo)
    ]
    [
        [
            "CVEGEO",
            "entidad",
            "municipio",
            "POBTOT_CENSO_2020",
            "GRAPROES_2020"
        ]
    ]
    .sort_values("CVEGEO")
    .reset_index(drop=True)
)

print(
    "Municipios del Censo fuera de la muestra candidata:",
    len(detalle_censo_fuera_candidata)
)

display(detalle_censo_fuera_candidata)

Municipios del Censo fuera de la muestra candidata: 4


,CVEGEO,entidad,municipio,POBTOT_CENSO_2020,GRAPROES_2020
0,02006,Baja California,San Quintín,117568,7.83
1,04012,Campeche,Seybaplaya,15297,9.0
2,07125,Chiapas,Honduras de la Sierra,11650,6.17
3,17036,Morelos,Hueyapan,7855,7.41


### 3.9 Actualización de la muestra candidata después de incorporar Censo 2020

La auditoría de cobertura confirmó que los 2,465 municipios previamente seleccionados mediante DENUE 2020, DENUE 2025 y CONAPO también se encuentran disponibles en el Censo 2020.

Por tanto, la incorporación de la variable educativa `GRAPROES_2020` no reduce el tamaño de la muestra candidata.

Los cuatro municipios adicionales presentes en Censo 2020 se mantienen fuera de la muestra porque no pertenecen simultáneamente al universo común definido por DENUE y CONAPO.

La muestra candidata continúa con 2,465 municipios y todavía no se considera definitiva, ya que faltan las fuentes ILMM y CONEVAL.

In [52]:
muestra_candidata_censo = (
    muestra_candidata[
        muestra_candidata["CVEGEO"]
        .isin(claves_comunes_censo)
    ]
    .copy()
    .sort_values("CVEGEO")
    .reset_index(drop=True)
)

print(
    "Municipios en nueva muestra candidata:",
    f"{len(muestra_candidata_censo):,}"
)

print(
    "CVEGEO únicos:",
    f"{muestra_candidata_censo['CVEGEO'].nunique():,}"
)

print(
    "CVEGEO duplicados:",
    muestra_candidata_censo["CVEGEO"].duplicated().sum()
)

Municipios en nueva muestra candidata: 2,465
CVEGEO únicos: 2,465
CVEGEO duplicados: 0


In [53]:
################ 3.10 Validar que GRAPROES existe para los 2,465 candidatos
validacion_graproes_candidatos = (
    muestra_candidata_censo[["CVEGEO"]]
    .merge(
        censo_2020_municipal[
            [
                "CVEGEO",
                "GRAPROES_2020"
            ]
        ],
        on="CVEGEO",
        how="left",
        validate="one_to_one"
    )
)

print(
    "Municipios candidatos:",
    f"{len(validacion_graproes_candidatos):,}"
)

print(
    "GRAPROES_2020 faltantes:",
    validacion_graproes_candidatos[
        "GRAPROES_2020"
    ].isna().sum()
)

print(
    "GRAPROES_2020 mínimo:",
    validacion_graproes_candidatos[
        "GRAPROES_2020"
    ].min()
)

print(
    "GRAPROES_2020 máximo:",
    validacion_graproes_candidatos[
        "GRAPROES_2020"
    ].max()
)


Municipios candidatos: 2,465
GRAPROES_2020 faltantes: 0
GRAPROES_2020 mínimo: 3.4
GRAPROES_2020 máximo: 14.55


In [54]:
#######3.11 Guardar la muestra candidata actualizada
archivo_muestra_candidata_censo = (
    PROCESSED_DIR
    / "muestra_candidata_denue_conapo_censo.csv"
)

muestra_candidata_censo.to_csv(
    archivo_muestra_candidata_censo,
    index=False,
    encoding="utf-8-sig"
)

print(
    "Muestra candidata DENUE-CONAPO-Censo "
    "guardada correctamente."
)

print(archivo_muestra_candidata_censo)

Muestra candidata DENUE-CONAPO-Censo guardada correctamente.
c:\Users\nashe\Desktop\Econometría\PROYECTO\atractividad_comercial_Mexico\data\processed\muestra_candidata_denue_conapo_censo.csv


3.12 Cierre del Censo 2020

### Conclusión de la preparación de Censo 2020

La base municipal del Censo 2020 quedó construida y validada correctamente.

Resultados principales:

- Se obtuvieron 2,469 municipios con información censal.
- Cada municipio presenta una clave `CVEGEO` única.
- No se encontraron valores faltantes en la base municipal.
- `GRAPROES_2020` no presenta valores faltantes ni símbolos de confidencialidad a nivel municipal.
- El grado promedio de escolaridad municipal presenta valores entre 3.40 y 14.55.
- Los 2,465 municipios de la muestra candidata DENUE–CONAPO están presentes en Censo 2020.
- La incorporación de Censo no provoca pérdida de observaciones.
- Los 2,465 municipios candidatos cuentan con información de `GRAPROES_2020`.

Por tanto, `GRAPROES_2020` queda disponible como la variable explicativa educativa X2 del modelo econométrico.

La muestra continúa siendo provisional hasta incorporar ILMM 2020 y CONEVAL 2020.